# 06 - Post Analysis
## Feature Importance та PCA експерименти (2f)

У цьому notebook ми:
- Проаналізуємо важливість фічів
- Виконаємо PCA аналіз
- Порівняємо результати з різною кількістю компонент
- Зробимо додаткові експерименти

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.embeddings import load_embeddings
from src.train import load_model, split_data, train_model_with_gridsearch
from src.post_analysis import (
    analyze_feature_importance,
    plot_feature_importance,
    perform_pca_analysis,
    plot_explained_variance
)
from src.evaluate import evaluate_model, print_evaluation_results
from src.config import PROCESSED_DATA_PATH, MODELS_PATH

%matplotlib inline

## 1. Завантаження даних та моделей

In [ ]:
# Завантажуємо дані
df = pd.read_csv(PROCESSED_DATA_PATH / 'cleaned_data.csv')
embeddings = load_embeddings(str(PROCESSED_DATA_PATH / 'embeddings.npy'))

# Розділяємо на train/test
X = embeddings
y = df['label'].values
X_train, X_test, y_train, y_test = split_data(X, y)

# Завантажуємо модель
model = load_model(str(MODELS_PATH / 'logistic_regression.joblib'))

print(f"Розмір даних: {X.shape}")
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

## 2. Feature Importance Analysis

In [ ]:
# Аналізуємо важливість фічів
importance_df = analyze_feature_importance(
    model,
    X_test,
    y_test,
    n_repeats=10
)

print("\nТоп-10 найважливіших фічів:")
print(importance_df.head(10))

In [ ]:
# Візуалізуємо важливість фічів
plot_feature_importance(
    importance_df,
    top_n=30,
    save=True,
    filename='feature_importance.png'
)

## 3. PCA Analysis

In [ ]:
# Виконаємо PCA аналіз
pca, embeddings_pca = perform_pca_analysis(
    X_train,
    n_components=100,
    plot=True
)

In [ ]:
# Подивимось на кумулятивну пояснену варіацію
cumsum_variance = np.cumsum(pca.explained_variance_ratio_)

# Знайдемо кількість компонент для 90% та 95% варіації
n_components_90 = np.argmax(cumsum_variance >= 0.90) + 1
n_components_95 = np.argmax(cumsum_variance >= 0.95) + 1

print(f"\nКількість компонент для 90% варіації: {n_components_90}")
print(f"Кількість компонент для 95% варіації: {n_components_95}")

## 4. Експерименти з різною кількістю PCA компонент

In [ ]:
from sklearn.decomposition import PCA

# Тестуємо різну кількість компонент
n_components_list = [10, 20, 50, 100, 200]
results = []

for n_comp in n_components_list:
    print(f"\n{'='*60}")
    print(f"Тестуємо з {n_comp} PCA компонентами")
    print(f"{'='*60}")
    
    # PCA трансформація
    pca_temp = PCA(n_components=n_comp)
    X_train_pca = pca_temp.fit_transform(X_train)
    X_test_pca = pca_temp.transform(X_test)
    
    explained_var = pca_temp.explained_variance_ratio_.sum()
    print(f"Пояснена варіація: {explained_var:.4f}")
    
    # Навчання моделі
    model_pca = train_model_with_gridsearch(
        X_train_pca,
        y_train,
        model_name='logistic_regression',
        cv=3
    )
    
    # Оцінка
    y_pred = model_pca.predict(X_test_pca)
    metrics = evaluate_model(y_test, y_pred)
    
    results.append({
        'n_components': n_comp,
        'explained_variance': explained_var,
        **metrics
    })
    
    print_evaluation_results(metrics)

In [ ]:
# Візуалізуємо результати
results_df = pd.DataFrame(results)
print("\nРезультати експериментів з PCA:")
print(results_df)

In [ ]:
# Побудуємо графік
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy vs n_components
axes[0].plot(results_df['n_components'], results_df['accuracy'], 'o-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of PCA Components', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Accuracy vs Number of PCA Components', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# F1-score vs n_components
axes[1].plot(results_df['n_components'], results_df['f1_score'], 'o-', 
             linewidth=2, markersize=8, color='orange')
axes[1].set_xlabel('Number of PCA Components', fontsize=12)
axes[1].set_ylabel('F1 Score', fontsize=12)
axes[1].set_title('F1 Score vs Number of PCA Components', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()

from src.config import OUTPUTS_PATH
plt.savefig(OUTPUTS_PATH / 'pca_experiments.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Порівняння з оригінальними ембедингами

In [ ]:
# Оригінальна модель
y_pred_original = model.predict(X_test)
metrics_original = evaluate_model(y_test, y_pred_original)

print("Результати на оригінальних ембедингах:")
print_evaluation_results(metrics_original)

# Найкраща PCA модель
best_pca_result = results_df.loc[results_df['accuracy'].idxmax()]

print(f"\nНайкраща PCA модель ({int(best_pca_result['n_components'])} компонент):")
print(f"Accuracy: {best_pca_result['accuracy']:.4f}")
print(f"F1 Score: {best_pca_result['f1_score']:.4f}")
print(f"Пояснена варіація: {best_pca_result['explained_variance']:.4f}")

## 6. Висновки та рекомендації

In [ ]:
print("""\n=== ВИСНОВКИ ===""")
print(f"""
1. Feature Importance:
   - Проаналізовано важливість {len(importance_df)} фічів
   - Топ-10 фічів мають найбільший вплив на класифікацію

2. PCA Analysis:
   - {n_components_90} компонент пояснюють 90% варіації
   - {n_components_95} компонент пояснюють 95% варіації

3. Експерименти з PCA:
   - Найкраща кількість компонент: {int(best_pca_result['n_components'])}
   - Accuracy: {best_pca_result['accuracy']:.4f}
   - Зменшення розмірності може прискорити навчання
   - Незначна втрата якості при зменшенні розмірності

4. Рекомендації:
   - Для балансу швидкості та якості рекомендується {n_components_95} компонент
   - Оригінальні ембединги дають найкращу якість
   - PCA корисна для візуалізації та прискорення навчання
""")

## Висновки

✅ Проаналізовано важливість фічів для класифікації
✅ Виконано PCA аналіз з різною кількістю компонент
✅ Порівняно продуктивність моделей на різних представленнях даних
✅ Знайдено оптимальний баланс між розмірністю та якістю
✅ Надано рекомендації для подальшого використання